In [0]:
from pyspark.sql.functions import col, regexp_extract, regexp_replace, when, split, element_at, trim, size
Forzar guardado sin GPU
dbutils.widgets.text("catalog_param", "")
dbutils.widgets.text("source_schema_param", "")
dbutils.widgets.text("target_schema_param", "")
dbutils.widgets.text("raw_schema_param", "")
# 2. Capturar los valores inyectados por el YAML
catalog_name = dbutils.widgets.get("catalog_param")
source_schema = dbutils.widgets.get("source_schema_param")
target_schema = dbutils.widgets.get("target_schema_param")
raw_schema = dbutils.widgets.get("raw_schema_param")

df_bronze = spark.readStream.table("{catalog_name}.{source_schema}.inmuebles")
target_table = "{source_schema}.inmuebles"
catalog_name = "real_state_project"
spark.catalog.setCurrentCatalog(catalog_name)
base_landing_path = f"/Volumes/{catalog_name}/{target_schema}/checkpoint/"
silver_checkpoint_path = f"{base_landing_path}.checkpoints_silver_inmuebles"
#target_silver_table = f"{base_landing_path}.Silver_Inmuebles_Historico"

# 1. Función a prueba de balas (Optimizada para evitar choques de tipos NullType en Databricks)
def extraer_numero_seguro(columna, patron, grupo):
    texto = regexp_extract(columna, patron, grupo)
    texto_limpio = regexp_replace(texto, ",", "")
    # Si encuentra número, castea a int. Si no (cadena vacía), Spark asigna null automáticamente.
    return when(texto_limpio != "", texto_limpio).cast("int")

# 2. Extracción de métricas (Lógica original restaurada: el número siempre está a la izquierda)
df_silver_raw = df_bronze \
    .withColumn("Id_Inmueble", regexp_extract(col("Url"), r"-(\d+)\.html", 1)) \
    .withColumn("Moneda", regexp_extract(col("Text_raw"), r"(MN|USD)", 1)) \
    .withColumn("Precio", extraer_numero_seguro(col("Text_raw"), r"(MN|USD)\s+([\d,]+)(?!\s*Mantenimiento)", 2)) \
    .withColumn("Mantenimiento", extraer_numero_seguro(col("Text_raw"), r"(MN|USD)\s+([\d,]+)\s*Mantenimiento", 2)) \
    .withColumn("Area_m2", extraer_numero_seguro(col("Text_raw"), r"([\d,]+)\s*m²", 1)) \
    .withColumn("Recamaras", extraer_numero_seguro(col("Text_raw"), r"(\d+)\s*rec", 1)) \
    .withColumn("Banos", extraer_numero_seguro(col("Text_raw"), r"(\d+)\s*baño", 1)) \
    .withColumn("Estacionamientos", extraer_numero_seguro(col("Text_raw"), r"(\d+)\s*estac", 1))

# 3. Tratamiento de Direcciones (Ahora soporta saltos de línea de Linux \n y Windows \r\n)
df_silver_address = df_silver_raw \
    .withColumn("Calle_Colonia", trim(regexp_extract(col("Text_raw"), r"(?:m²|rec\.|baños?|estac\.)[^\r\n]*[\r\n]+([^\r\n]+)", 1))) \
    .withColumn("Mun_Edo_Raw", trim(regexp_extract(col("Text_raw"), r"(?:m²|rec\.|baños?|estac\.)[^\r\n]*[\r\n]+[^\r\n]+[\r\n]+([^\r\n]+)", 1))) \
    .withColumn("Array_Ubicacion", split(col("Mun_Edo_Raw"), ",")) \
    .withColumn("Municipio_Delegacion", when(size(col("Array_Ubicacion")) >= 1, trim(element_at(col("Array_Ubicacion"), 1)))) \
    .withColumn("Estado", when(size(col("Array_Ubicacion")) >= 2, trim(element_at(col("Array_Ubicacion"), 2)))) \
    .drop("Array_Ubicacion", "Mun_Edo_Raw")

# 4. Reglas de Negocio (Eliminamos temporalmente la regla de "lote" para evitar falsos positivos)
df_silver_clean = df_silver_address \
    .withColumn("Mantenimiento", when((col("Mantenimiento") >= (col("Precio") * 0.10)), None).otherwise(col("Mantenimiento"))) \
    .withColumn("Area_m2", when(col("Area_m2") > 1200, None).otherwise(col("Area_m2"))) \
    .dropDuplicates(["Id_Inmueble","extraction_date"])

# 5. Auditoría visual de los campos clave
#display(df_silver_clean.select(
 #   "Id_Inmueble", "Area_m2", "Recamaras", "Banos", "Estacionamientos", 
  #  "Calle_Colonia", "Municipio_Delegacion", "Estado"
#))

In [0]:
import pyspark.sql.functions as F

# ==============================================================================
# 1. EL TRUCO DEL ESQUEMA FORZADO
# Le decimos a Spark que solo nos interesan las primeras 5 columnas.
# Columna 3 (4ta posición) es Municipio. Columna 4 (5ta posición) es Estado.
# ==============================================================================
esquema_forzado = "_c0 STRING, _c1 STRING, _c2 STRING, municipio_oficial STRING, estado_oficial STRING"

# ==============================================================================
# 2. LECTURA ESTRICTA
# El formato CSV sí respeta la codificación windows-1252 (ANSI)
# ==============================================================================
df_sepomex_raw = spark.read \
    .format("csv") \
    .schema(esquema_forzado) \
    .option("header", "false") \
    .option("sep", "|") \
    .option("encoding", "windows-1252") \
    .load("/Volumes/{catalog_name}/{raw_schema}/catalogo_maestro/CPdescarga.txt")

# ==============================================================================
# 3. LIMPIEZA INMEDIATA
# ==============================================================================
df_catalogo_maestro = df_sepomex_raw \
    .select("municipio_oficial", "estado_oficial") \
    .distinct() \
    .filter(
        F.col("municipio_oficial").isNotNull() & 
        (F.col("municipio_oficial") != "") &
        (F.col("municipio_oficial") != "D_mnpio") 
    )

print("📊 Catálogo Maestro con Acentos Perfectos:")
#display(df_catalogo_maestro)

In [0]:
from pyspark.sql.functions import col, when, lower, trim, broadcast

# =================================================================================
# 6. CRUCE MAESTRO CON SEPOMEX (Mejorado con Normalización y Búsqueda Flexible)
# =================================================================================

# 6.1 Preparamos el catálogo
cat_df = df_catalogo_maestro \
    .dropDuplicates(["municipio_oficial"]) \
    .select(
        col("municipio_oficial").alias("cat_mun"),
        col("estado_oficial").alias("cat_edo")
    )

# 6.2 Diccionario de Normalización (Limpiamos la basura ANTES de cruzar)
# Convertimos las "ciudades turísticas/rebeldes" a sus verdaderos municipios y estados
df_silver_norm = df_silver_clean.withColumn(
    "mun_norm",
    when(lower(col("Estado")).contains("cancún") | lower(col("Estado")).contains("cancun"), "Benito Juárez")
    .when(lower(col("Estado")).contains("playa del carmen") | lower(col("Estado")).contains("riviera maya"), "Solidaridad")
    .otherwise(col("Municipio_Delegacion"))
).withColumn(
    "edo_norm",
    when(lower(col("Estado")).contains("cancún") | lower(col("Estado")).contains("cancun"), "Quintana Roo")
    .when(lower(col("Estado")).contains("playa del carmen") | lower(col("Estado")).contains("riviera maya"), "Quintana Roo")
    .when(lower(col("Estado")).contains("pachuca"), "Pachuca de Soto") # Lo forzamos al nombre oficial
    .otherwise(col("Estado"))
)

# 6.3 Cruces Flexibles usando "StartsWith" en lugar de "=="
# Esto permite que "Pachuca" haga match con "Pachuca de Soto" y "Mérida Centro" con "Mérida"

# Escenario A segurizado contra nulos/vacíos
condicion_a = (col("edo_norm") != "") & (col("edo_norm").isNotNull()) & (
    lower(trim(col("cat_mun"))).startswith(lower(trim(col("edo_norm")))) |
    lower(trim(col("edo_norm"))).startswith(lower(trim(col("cat_mun"))))
)

df_join_1 = df_silver_norm.join(
    broadcast(cat_df),
    condicion_a,
    "left"
).withColumnRenamed("cat_mun", "mun_match_1") \
 .withColumnRenamed("cat_edo", "edo_match_1")

# Escenario B segurizado
condicion_b = (col("mun_norm") != "") & (col("mun_norm").isNotNull()) & (
    lower(trim(col("cat_mun"))).startswith(lower(trim(col("mun_norm")))) |
    lower(trim(col("mun_norm"))).startswith(lower(trim(col("cat_mun"))))
)

df_join_2 = df_join_1.join(
    broadcast(cat_df),
    condicion_b,
    "left"
).withColumnRenamed("cat_mun", "mun_match_2") \
 .withColumnRenamed("cat_edo", "edo_match_2")

# 6.4 Resolución de quién ganó
df_silver_enriched = df_join_2.withColumn(
    "Municipio_Oficial",
    when(col("mun_match_1").isNotNull(), col("mun_match_1"))
    .when(col("mun_match_2").isNotNull(), col("mun_match_2"))
    .otherwise(col("mun_norm")) # Si todo falla, dejamos el normalizado
).withColumn(
    "Estado_Oficial",
    when(col("edo_match_1").isNotNull(), col("edo_match_1"))
    .when(col("edo_match_2").isNotNull(), col("edo_match_2"))
    .otherwise(col("edo_norm"))
).drop(
    # Limpieza del esquema
    "mun_match_1", "edo_match_1", "mun_match_2", "edo_match_2", 
    "Municipio_Delegacion", "Estado", "mun_norm", "edo_norm"
)

# 7. Visualización final
#display(df_silver_enriched.select(
 #   "Id_Inmueble", "Precio", "Mantenimiento", "Area_m2", 
  #  "Recamaras", "Banos", "Estacionamientos", 
   # "Calle_Colonia", "Municipio_Oficial", "Estado_Oficial"
#))

In [0]:
from pyspark.sql.functions import col, when, lit, broadcast

# =================================================================================
# 7. FILTRO ESTRICTO DE CALIDAD OPTIMIZADO (CORREGIDO)
# =================================================================================

# 1. Preparamos el catálogo y RENOMBRAMOS la columna para evitar ambigüedad
df_estados_validos = df_catalogo_maestro \
    .select(col("estado_oficial").alias("cat_estado")) \
    .distinct() \
    .filter(col("cat_estado").isNotNull()) \
    .withColumn("es_valido", lit(True))

# 2. Usamos el LEFT JOIN con Broadcast usando los nombres distintos
df_silver_joined = df_silver_enriched.join(
    broadcast(df_estados_validos),
    df_silver_enriched["Estado_Oficial"] == df_estados_validos["cat_estado"],
    "left"
)

# 3. Aplicamos la regla: Ahora Spark sabe exactamente cuál Estado_Oficial tomar
df_silver_blindado0 = df_silver_joined.withColumn(
    "Estado_Oficial",
    when(col("es_valido").isNotNull(), col("Estado_Oficial"))
    .otherwise(lit("No Definido"))
)

# 4. Limpiamos también el Municipio si el Estado falló, y quitamos columnas temporales
df_silver_blindado1 = df_silver_blindado0.withColumn(
    "Municipio_Oficial",
    when(col("Estado_Oficial") == "No Definido", lit("No Definido"))
    .otherwise(col("Municipio_Oficial"))
).drop("cat_estado", "es_valido") 
# Quitamos el catálogo temporal

df_silver_blindado = df_silver_blindado1.dropDuplicates(["Id_Inmueble"])
query_silver = df_silver_blindado.writeStream \
        .format("delta") \
        .outputMode("append") \
        .option("checkpointLocation", silver_checkpoint_path) \
        .option("mergeSchema", "true") \
        .trigger(availableNow=True) \
        .toTable(target_table)

query_silver.awaitTermination()
# 5. Verificación visual
#display(df_silver_blindado.groupBy("Estado_Oficial").count().orderBy("Estado_Oficial"))